#  AA-CLIP Baseline Reproduction — MVTec-AD & VisA


## 1. Clone repo AA-CLIP và cài dependencies

In [1]:
import os
os.chdir('/kaggle/working')

!git clone https://github.com/Mwxinnn/AA-CLIP.git
os.chdir('/kaggle/working/AA-CLIP')
!ls

Cloning into 'AA-CLIP'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 111 (delta 15), reused 9 (delta 9), pack-reused 87 (from 2)
Receiving objects: 100% (111/111), 5.23 MiB | 10.16 MiB/s, done.
Resolving deltas: 100% (31/31), done.
dataset		  LICENSE  pic	      requirements.txt	scripts.sh  train.py
forward_utils.py  model    README.md  results		test.py     utils.py


In [2]:
# Cài requirements
!pip install -q open_clip_torch einops timm ftfy regex ipdb
!pip install -q -r requirements.txt 2>/dev/null || echo 'requirements.txt done (some may already be installed)'
print('Dependencies installed')

import os, subprocess, re, shutil, json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
requirements.txt done (some may already be installed)
Dependencies installed


In [3]:
SHOT = 0

paper_results = {
    'MVTec-AD': {'image_auroc': 91.5, 'pixel_auroc': 97.8},  # kết quả từ paper gốc
    'VisA':     {'image_auroc': 85.1, 'pixel_auroc': 98.1},
}

In [4]:
import subprocess

# Tìm và patch tất cả file còn path cũ
result = subprocess.run(
    ['grep', '-r', '/data/wenxinma', '/kaggle/working/AA-CLIP', '--include=*.py', '-l'],
    capture_output=True, text=True
)

files = [f for f in result.stdout.strip().split('\n') if f]
if not files:
    print(' Không có file nào cần patch')
else:
    for fpath in files:
        with open(fpath) as f:
            content = f.read()
        content = content.replace('/data/wenxinma/data/mvtec_ad', '/kaggle/working/AA-CLIP/data/mvtec_ad')
        content = content.replace('/data/wenxinma/data/VisA_20220922', '/kaggle/working/AA-CLIP/data/VisA_20220922')
        content = content.replace('/data/wenxinma', '/kaggle/working/AA-CLIP')
        with open(fpath, 'w') as f:
            f.write(content)
        print(f' Patched: {fpath}')

 Patched: /kaggle/working/AA-CLIP/dataset/constants.py


# 2. Download OpenCLIP ViT-L-14-336px weights


In [5]:
import open_clip
import os

os.makedirs('/kaggle/working/AA-CLIP/model/weights', exist_ok=True)

# Download và cache OpenCLIP ViT-L-14-336
print('Downloading OpenCLIP ViT-L-14-336 weights (this may take ~5 min)...')
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-L-14-336',
    pretrained='openai',
    cache_dir='/kaggle/working/AA-CLIP/model/weights'
)
print('OpenCLIP weights downloaded')

# Kiểm tra file
!find /kaggle/working/AA-CLIP/model/weights -name '*.pt' -o -name '*.bin' 2>/dev/null | head -5

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


OpenCLIP weights downloaded


## 3. Chuẩn bị MVTec AD Dataset


In [6]:
import os

MVTEC_PATH = '/kaggle/input/datasets/ipythonx/mvtec-ad'             
# Kiểm tra
for name, path in [('MVTec-AD', MVTEC_PATH)]:
    if os.path.exists(path):
        cats = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
        print(f' {name} found at {path} — {len(cats)} categories: {cats[:5]}...')
    else:
        print(f'  {name} NOT found at {path} — xem cell bên dưới để tải')

 MVTec-AD found at /kaggle/input/datasets/ipythonx/mvtec-ad — 15 categories: ['wood', 'screw', 'metal_nut', 'capsule', 'hazelnut']...


In [7]:
# [TÙY CHỌN] Download MVTec-AD nếu chưa có
# Uncomment nếu cần

# !pip install -q kaggle
# !kaggle datasets download -d ipythonx/mvtec-ad --unzip -p /kaggle/working/data/
# MVTEC_PATH = '/kaggle/working/data/mvtec_anomaly_detection'

# Hoặc dùng gdown để tải VisA
# !pip install -q gdown
# !gdown --fuzzy https://drive.google.com/... -O /kaggle/working/data/

print('Uncomment cell trên để tải dataset nếu chưa có')

Uncomment cell trên để tải dataset nếu chưa có


# Symlink

In [8]:
import os

os.makedirs('/kaggle/working/AA-CLIP/data', exist_ok=True)

mvtec_link = '/kaggle/working/AA-CLIP/data/mvtec_ad' 
MVTEC_PATH = '/kaggle/input/datasets/ipythonx/mvtec-ad'

if not os.path.exists(mvtec_link):
    os.symlink(MVTEC_PATH, mvtec_link)
    print(' MVTec linked:', mvtec_link)
else:
    print(' Already exists!')

print('Folders:', os.listdir(mvtec_link)[:5])

# # Tạo symlink cho VisA
import os

VISA_PATH = '/kaggle/input/datasets/tensura3607/amazon-visa-anomaly'
visa_link = '/kaggle/working/AA-CLIP/data/VisA_20220922'

if not os.path.exists(visa_link):
    os.symlink(VISA_PATH, visa_link)
    print(' VisA linked!')
else:
    print(' Already exists!')

print('Folders:', os.listdir(visa_link)[:5])

 MVTec linked: /kaggle/working/AA-CLIP/data/mvtec_ad
Folders: ['wood', 'screw', 'metal_nut', 'capsule', 'readme.txt']
 VisA linked!
Folders: ['pcb2', 'pipe_fryum', 'candle', 'LICENSE-DATASET', 'fryum']


## 4. Chỉnh sửa constants.py — trỏ đúng data path

In [9]:
constants_path = '/kaggle/working/AA-CLIP/dataset/constants.py'

with open(constants_path, 'r') as f:
    content = f.read()

# Fix BASE_PATH đúng - không có /data ở cuối vì DATA_PATH đã có f"{BASE_PATH}/data/..."
new_content = content.replace(
    'BASE_PATH = "/kaggle/working/AA-CLIP/data"',
    'BASE_PATH = "/kaggle/working/AA-CLIP"'
)

with open(constants_path, 'w') as f:
    f.write(new_content)

print(' Fixed!')

# Verify
with open(constants_path) as f:
    for line in f:
        if 'BASE_PATH' in line or 'mvtec' in line:
            print(line, end='')

 Fixed!
BASE_PATH = "/kaggle/working/AA-CLIP"
    "Brain": f"{BASE_PATH}/data/MedAD/Brain_AD",
    "Liver": f"{BASE_PATH}/data/MedAD/Liver_AD",
    "Retina": f"{BASE_PATH}/data/MedAD/Retina_RESC_AD",
    "Colon_clinicDB": f"{BASE_PATH}/data/Colon/CVC-ClinicDB",
    "Colon_colonDB": f"{BASE_PATH}/data/Colon/CVC-ColonDB",
    "Colon_cvc300": f"{BASE_PATH}/data/Colon/CVC-300",
    "Colon_Kvasir": f"{BASE_PATH}/data/Colon/Kvasir",
    "BTAD": f"{BASE_PATH}/data/BTech_Dataset_transformed",
    "MPDD": f"{BASE_PATH}/data/MPDD",
    "MVTec": f"{BASE_PATH}/data/mvtec_ad",
    "VisA": f"{BASE_PATH}/data/VisA_20220922",


In [10]:
patch = r'''

import os
import torch
import open_clip
from typing import List, Optional, Union
# Lưu ý: Đảm bảo các hàm này có tồn tại trong môi trường của bạn
from .model import build_model_from_openai_state_dict, convert_weights_to_lp, get_cast_dtype

__all__ = ["list_openai_models", "load_openai_model"]

def load_openai_model(
        name: str,
        precision: Optional[str] = None,
        device: Optional[Union[str, torch.device]] = None,
        jit: bool = True,
        cache_dir: Optional[str] = None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if precision is None:
        precision = 'fp32' if device == 'cpu' else 'fp16'

    print(f"[Patched] Loading ViT-L-14-336 via open_clip...")
    
    # Để thực sự dùng safetensors, bạn nên trỏ trực tiếp tới file nếu open_clip version hỗ trợ
    model, _, _ = open_clip.create_model_and_transforms(
        'ViT-L-14-336',
        pretrained='openai', 
        cache_dir='/kaggle/working/AA-CLIP/model/weights'
    )

    model = model.to(device)
    if precision == 'fp32' or precision.startswith('amp'):
        model.float()
    elif precision == 'fp16':
        model.half()
    elif precision == 'bf16':
        model.to(torch.bfloat16)

    return model
'''

with open('/kaggle/working/AA-CLIP/model/openai.py', 'w', encoding='utf-8') as f:
    f.write(patch)

print(' openai.py patched successfully!')

 openai.py patched successfully!


## 5. TRAINING — Huấn luyện AA-CLIP trên MVTec


In [11]:
import os, subprocess
os.chdir('/kaggle/working/AA-CLIP')

SAVE_PATH = '/kaggle/working/AA-CLIP/checkpoints/mvtec_fullshot'
os.makedirs(SAVE_PATH, exist_ok=True)

train_cmd = [
    'python', 'train.py',
    '--dataset', 'MVTec',
    '--training_mode', 'full_shot',
    '--shot', '0',
    '--save_path', SAVE_PATH,
    '--image_batch_size', '4',   
    '--text_batch_size', '4',    
]

print(f'Running: {" ".join(train_cmd)}')
print('='*60)

result = subprocess.run(train_cmd, capture_output=False, text=True)
print('Return code:', result.returncode)

Running: python train.py --dataset MVTec --training_mode full_shot --shot 0 --save_path /kaggle/working/AA-CLIP/checkpoints/mvtec_fullshot --image_batch_size 4 --text_batch_size 4
[Patched] Loading ViT-L-14-336 via open_clip...
[Patched] Loading ViT-L-14-336 via open_clip...


  0%|          | 0/432 [00:01<?, ?it/s]
Traceback (most recent call last):
  File "/kaggle/working/AA-CLIP/train.py", line 361, in <module>
    main()
  File "/kaggle/working/AA-CLIP/train.py", line 345, in main
    model = train_image_adapter(
            ^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/AA-CLIP/train.py", line 145, in train_image_adapter
    patch_features, det_feature = model(image)
                                  ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/AA-CLIP/model/adapter.py", line 91, in forward
    x, attn = self.image_encoder.transformer.resblocks[i](x, attn_mask=None)
              ^^^^^^^^

Return code: 1


## 6. EVALUATION — Đánh giá trên MVTec-AD và VisA

In [12]:
# os.chdir('/kaggle/working/AA-CLIP')

# # =========================================================
# # Evaluate trên MVTec-AD
# # =========================================================
# print('=== Evaluating on MVTec-AD ===')
# !python test.py \
#     --save_path /kaggle/working/AA-CLIP/checkpoints/visa_0shot \
#     --dataset mvtec \
#     2>&1 | tee /kaggle/working/eval_mvtec.log

In [13]:
import os, subprocess
os.chdir('/kaggle/working/AA-CLIP')

print('=== Evaluating on VisA ===')
with open('/kaggle/working/eval_visa.log', 'w') as log_f:
    result = subprocess.run([
        'python', 'test.py',
        '--dataset', 'VisA',
        '--save_path', '/kaggle/working/AA-CLIP/checkpoints/mvtec_fullshot',
    ], stdout=log_f, stderr=log_f, text=True)
print('Return code:', result.returncode)

=== Evaluating on VisA ===
Return code: 1


## 7. Phân tích kết quả — So sánh với paper

In [14]:
def parse_auroc_from_log(log_path):
    results = {'image_auroc': None, 'pixel_auroc': None}
    
    if not os.path.exists(log_path):
        print(f"[WARN] Không tìm thấy log: {log_path}")
        return results
    
    with open(log_path) as f:
        text = f.read()
    
    img_match = re.search(r'[Ii]mage.*?AUROC[:\s]+([\d.]+)', text)
    if img_match:
        results['image_auroc'] = float(img_match.group(1))
    
    pix_match = re.search(r'[Pp]ixel.*?AUROC[:\s]+([\d.]+)', text)
    if pix_match:
        results['pixel_auroc'] = float(pix_match.group(1))

    if not img_match and not pix_match:
        print(f"[WARN] Không parse được AUROC. Nội dung log cuối:\n{text[-2000:]}")
    
    return results

# Parse log VisA
visa_res = parse_auroc_from_log('/kaggle/working/eval_visa.log')

print('=== Kết quả ===')
print(f"{'Dataset':<12} | {'Image AUROC':>12} | {'Pixel AUROC':>12}")
print('-'*42)
img = f"{visa_res['image_auroc']:.1f}%" if visa_res['image_auroc'] else 'N/A'
pix = f"{visa_res['pixel_auroc']:.1f}%" if visa_res['pixel_auroc'] else 'N/A'
print(f"{'VisA':<12} | {img:>12} | {pix:>12}")
print()
print('Ghi chú: Train trên MVTec, test cross-domain trên VisA')

[WARN] Không parse được AUROC. Nội dung log cuối:
[Patched] Loading ViT-L-14-336 via open_clip...
Traceback (most recent call last):
  File "/kaggle/working/AA-CLIP/test.py", line 253, in <module>
    main()
  File "/kaggle/working/AA-CLIP/test.py", line 173, in main
    assert len(files) > 0, "image adapter checkpoint not found"
           ^^^^^^^^^^^^^^
AssertionError: image adapter checkpoint not found

=== Kết quả ===
Dataset      |  Image AUROC |  Pixel AUROC
------------------------------------------
VisA         |          N/A |          N/A

Ghi chú: Train trên MVTec, test cross-domain trên VisA


## 8. Lưu checkpoints và kết quả


In [15]:
import shutil, json

output_dir = '/kaggle/working/output_final'
os.makedirs(output_dir, exist_ok=True)

# Copy checkpoints
ckpt_src = '/kaggle/working/AA-CLIP/checkpoints'
if os.path.exists(ckpt_src):
    shutil.copytree(ckpt_src, f'{output_dir}/checkpoints', dirs_exist_ok=True)
    print(' Checkpoints saved')

# Chỉ copy log VisA 
for log in ['eval_visa.log']:
    src = f'/kaggle/working/{log}'
    if os.path.exists(src):
        shutil.copy(src, output_dir)
        print(f' {log} saved')

# Lưu summary JSON
summary = {
    'model': 'AA-CLIP (CVPR 2025)',
    'backbone': 'OpenCLIP ViT-L-14-336',
    'training_dataset': 'MVTec-AD',   
    'shot': SHOT,
    'results': {
        'VisA (cross-domain)': visa_res  
    },
    'paper_results': paper_results
}

with open(f'{output_dir}/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f' Summary saved to {output_dir}/results_summary.json')

print()
print('Output files:')
!ls -lh /kaggle/working/output_final/

 Checkpoints saved
 eval_visa.log saved
 Summary saved to /kaggle/working/output_final/results_summary.json

Output files:
total 12K
drwxr-xr-x 3 root root 4.0K May  8 14:24 checkpoints
-rw-r--r-- 1 root root  359 May  8 16:32 eval_visa.log
-rw-r--r-- 1 root root  407 May  8 16:32 results_summary.json
